# PhysAR-IDS -- NB4b (v6): Attack 2 -- how much perturbation budget breaks "physics immunity"?

NB4 found that at a cumulative budget of **0.1 sigma** nothing (PPO, random, PGD) evades ransomware or reverseshell, while the **unconstrained** PPO (ceiling 3 sigma) evades 100% of them in 2-3 steps. This notebook finds where between those two the immunity breaks.

**Design.** The same black-box PPO as NB4 (same environment, reward, algorithm, 50,000 timesteps, same CPU oracle over the Optuna-tuned XGBoost, same 5 seeds), trained and scored **separately at each cumulative budget** in {0.2, 0.3, 0.5, 1.0} sigma; 0.1 sigma is re-scored from the saved NB4 agents (and checked against the NB4 numbers). At every budget the two non-learning references are re-run too: random perturbation with the same 10 queries, and surrogate-transfer PGD on the cyber features. The per-step clip stays at 0.3 sigma, so budgets above 0.3 sigma need more than one step.

**Questions answered**
1. At what budget does each class start to evade (first budget with mean evasion > 5%, and > 50%)?
2. Does PPO beat random search as the budget grows (does *learning* start to matter)?
3. How far outside the training range do the adversarial cyber values go at each budget (realism)? A budget at which evasion appears only because values leave the observed range is a weaker finding than one that stays inside it.

**Runtime.** 4 budgets x 5 seeds = 20 PPO trainings of ~7 min each (~2.5 h); resumable. To do a quick pass first, set `SEEDS = [42]` in the setup cell (~30 min); files are fingerprinted per seed.
**Environment:** same as NB4 (`stable-baselines3 sb3-contrib gymnasium`, installed from a terminal).

## 0. Setup

In [ ]:
import os, sys, random, time, json, hashlib, shutil, platform, warnings
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
try:
    from sb3_contrib import RecurrentPPO
    from stable_baselines3.common.callbacks import CheckpointCallback
except ImportError as e:
    raise ImportError('Install in a terminal (not in a cell): pip install stable-baselines3 sb3-contrib gymnasium --no-deps') from e
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False

# ---- PPO / environment settings (unchanged from the reference design) ----
EPS_BOUND = 0.3; MAX_STEPS = 10
EVASION_REWARD = 10.0; STEP_PENALTY_SCALE = 0.01; TIMEOUT_PENALTY = -1.0
TOTAL_TIMESTEPS = 50000; CKPT_FREQ = 5000
EPS_TOTAL_BUDGET = 0.1
BUDGETS = [0.2, 0.3, 0.5, 1.0]      # new budgets (sigma); 0.1 is the NB4 reference
REF_BUDGET = 0.1
ALL_BUDGETS = [REF_BUDGET] + BUDGETS
EVAL_CHUNK = 8192

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} fit backend: {BACKEND_NAME}')
import sb3_contrib, stable_baselines3
print(f'sb3-contrib {sb3_contrib.__version__} | stable-baselines3 {stable_baselines3.__version__} | gymnasium {gym.__version__}')
print('Working directory:', os.getcwd())

## 1. Load v6 data and the NB2 tuned configuration

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

atk_tr = y_train != 0; X_atk_train, y_atk_train = X_train[atk_tr], y_train[atk_tr]
atk_te = y_test != 0;  X_atk_test,  y_atk_test  = X_test[atk_te],  y_test[atk_te]
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
print(f'Features {N_FEAT} | attacker-controlled (cyber) {N_CYBER} | training pool {X_atk_train.shape} | evaluation rows {X_atk_test.shape} in {len(np.unique(test_burst))} attack bursts')
print('Training pool class counts:', np.bincount(y_atk_train)[1:], '| evaluation class counts:', np.bincount(y_atk_test)[1:])
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. Oracle, environment and batched evaluator (identical to NB4)

In [ ]:
class Oracle:
    """Hard-label black-box oracle over the fitted XGBoost classifier."""
    def __init__(self, clf): self.clf = clf; self.n_queries = 0
    def predict(self, X):
        X = np.ascontiguousarray(X, dtype=np.float32); self.n_queries += len(X)
        return self.clf.predict(X)

def apply_action(x_cur, x_orig, action, eps_bound=EPS_BOUND, eps_total=None):
    """In place on x_cur[:, CYBER_IDX]; returns the applied (post-clamp) perturbation."""
    a = np.clip(action, -eps_bound, eps_bound).astype(np.float32)
    old = x_cur[:, CYBER_IDX].copy(); prop = old + a
    if eps_total is not None:
        o = x_orig[:, CYBER_IDX]; prop = o + np.clip(prop - o, -eps_total, eps_total)
    x_cur[:, CYBER_IDX] = prop
    return (prop - old).astype(np.float32)

def build_obs(x_cur, last_applied, last_pred, step):
    oh = np.zeros((len(x_cur), 6), dtype=np.float32); oh[np.arange(len(x_cur)), last_pred] = 1.0
    return np.concatenate([x_cur.astype(np.float32), last_applied.astype(np.float32), oh, np.full((len(x_cur), 1), step / MAX_STEPS, dtype=np.float32)], 1)

class IDSEvasionEnv(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, X_pool, y_pool, oracle, eps_bound=EPS_BOUND, max_steps=MAX_STEPS, seed=None, eps_total_budget=None):
        super().__init__()
        self.X_pool, self.y_pool, self.oracle = X_pool, y_pool, oracle
        self.eps_bound, self.max_steps, self.eps_total_budget = eps_bound, max_steps, eps_total_budget
        self.rng = np.random.RandomState(seed)
        self.action_space = spaces.Box(low=-eps_bound, high=eps_bound, shape=(N_CYBER,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(N_FEAT + N_CYBER + 6 + 1,), dtype=np.float32)
    def _obs(self): return build_obs(self.x_current[None], self.last_applied[None], np.array([self.last_pred]), self.step_count)[0]
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = self.rng.randint(0, len(self.X_pool))
        self.x_orig = self.X_pool[idx].copy(); self.x_current = self.x_orig.copy(); self.y_true = self.y_pool[idx]
        self.step_count = 0; self.last_applied = np.zeros(N_CYBER, dtype=np.float32)
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); return self._obs(), {}
    def step(self, action):
        applied = apply_action(self.x_current[None], self.x_orig[None], np.asarray(action, dtype=np.float32)[None], self.eps_bound, self.eps_total_budget)[0]
        self.last_applied = applied; self.step_count += 1
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); pen = STEP_PENALTY_SCALE * float(np.sum(applied ** 2))
        terminated = truncated = False
        if self.last_pred == 0: reward = EVASION_REWARD - pen; terminated = True
        elif self.step_count >= self.max_steps: reward = TIMEOUT_PENALTY - pen; truncated = True
        else: reward = -pen
        return self._obs(), reward, terminated, truncated, {'evaded': self.last_pred == 0}

def run_batched(policy, X0, oracle, eps_total, mode='ppo', rng=None):
    """Run one episode per row of X0 in parallel (10 steps max). mode: 'ppo' (deterministic policy) or 'random'."""
    N = len(X0); x_cur = X0.copy(); last_applied = np.zeros((N, N_CYBER), dtype=np.float32)
    last_pred = np.concatenate([oracle.predict(X0[i:i + 65536]) for i in range(0, N, 65536)]).astype(np.int64); clean_pred = last_pred.copy()
    active = np.ones(N, bool); evaded = np.zeros(N, bool); steps = np.zeros(N, np.int64)
    if mode == 'ppo': h = np.zeros((1, N, 64), np.float32); c = np.zeros((1, N, 64), np.float32)
    for step in range(MAX_STEPS):
        idx = np.where(active)[0]
        if len(idx) == 0: break
        obs = build_obs(x_cur[idx], last_applied[idx], last_pred[idx], step)
        if mode == 'ppo':
            act = np.zeros((len(idx), N_CYBER), np.float32)
            for s in range(0, len(idx), EVAL_CHUNK):
                sl = idx[s:s + EVAL_CHUNK]
                a, (hn, cn) = policy.predict(obs[s:s + EVAL_CHUNK], state=(h[:, sl], c[:, sl]), episode_start=np.full(len(sl), step == 0), deterministic=True)
                act[s:s + EVAL_CHUNK] = a; h[:, sl] = hn; c[:, sl] = cn
        else:
            act = rng.uniform(-EPS_BOUND, EPS_BOUND, size=(len(idx), N_CYBER)).astype(np.float32)
        sub = x_cur[idx]; applied = apply_action(sub, X0[idx], act, EPS_BOUND, eps_total); x_cur[idx] = sub; last_applied[idx] = applied
        pred = np.concatenate([oracle.predict(sub[i:i + 65536]) for i in range(0, len(sub), 65536)]).astype(np.int64); last_pred[idx] = pred
        ev = pred == 0; evaded[idx[ev]] = True; steps[idx[ev]] = step + 1; active[idx[ev]] = False
    return {'evaded': evaded, 'steps': steps, 'x_final': x_cur, 'pred_final': last_pred, 'clean_pred': clean_pred}

def per_class(vec_bool):
    return np.array([vec_bool[y_atk_test == c].mean() for c in range(1, 6)])
def mean_steps(res):
    return np.array([res['steps'][(y_atk_test == c) & res['evaded']].mean() if ((y_atk_test == c) & res['evaded']).any() else np.nan for c in range(1, 6)])
def summarise(res):
    disp = np.abs(res['x_final'][:, CYBER_IDX] - X_atk_test[:, CYBER_IDX])
    _ex = lambda Z: np.maximum(np.maximum(FEAT_LO[CYBER_IDX] - Z[:, CYBER_IDX], Z[:, CYBER_IDX] - FEAT_HI[CYBER_IDX]), 0).max(1)
    exc = np.maximum(_ex(res['x_final']) - _ex(X_atk_test), 0)    # sigma ADDED beyond what the clean test row already had outside the training min/max
    return {'per_class_evasion': per_class(res['evaded']), 'overall_evasion': float(res['evaded'].mean()), 'mean_steps_to_evade': mean_steps(res),
            'mean_queries': float(1 + np.where(res['evaded'], res['steps'], MAX_STEPS).mean()),
            'attack_f1': float(f1_score(y_atk_test, res['pred_final'], average='weighted')),
            'max_abs_disp_mean': float(disp.max(1).mean()), 'max_abs_disp_max': float(disp.max()), 'mean_excess_beyond_train_range': float(exc.mean()), 'frac_rows_beyond_range_0.5sigma': float((exc > 0.5).mean()),
            'evaded': res['evaded'].copy()}
print('environment + evaluator defined; cyber features controlled:', N_CYBER)

## 3. Training / loading an agent (resumable, fingerprinted)

In [ ]:
def fp_of(*parts): return hashlib.sha256(repr(parts).encode()).hexdigest()[:16]
def model_fp(oracle, X_probe): return hashlib.sha256(np.asarray(oracle.predict(X_probe)).tobytes()).hexdigest()[:16]
def latest_ckpt(d):
    if not os.path.isdir(d): return None
    c = [f for f in os.listdir(d) if f.endswith('_steps.zip')]
    if not c: return None
    best = max(c, key=lambda f: int(f.rsplit('_steps.zip', 1)[0].rsplit('_', 1)[1])); return os.path.join(d, best)

def get_agent(variant, seed, oracle):      # variant = total budget (float)
    eps_total = float(variant); name = f'ppo_attack2_budget{str(variant).replace(".", "p")}_{PIPELINE_VERSION}_seed{seed}'   # no '.' in the name: SB3 only appends .zip when there is no extension
    fp = fp_of(model_fp(oracle, X_atk_train[:300]), EPS_BOUND, MAX_STEPS, EVASION_REWARD, STEP_PENALTY_SCALE, TIMEOUT_PENALTY, TOTAL_TIMESTEPS, eps_total, seed, N_CYBER)
    if os.path.exists(name + '.zip') and os.path.exists(name + '.fp') and open(name + '.fp').read() == fp:
        print(f'    [{variant}/{seed}] finished agent found (fingerprint matches) -- loaded'); return RecurrentPPO.load(name), 0.0
    ckdir = f'_ckpt_{name}'; os.makedirs(ckdir, exist_ok=True); fpf = os.path.join(ckdir, '_fp.txt')
    env = IDSEvasionEnv(X_atk_train, y_atk_train, oracle, seed=seed, eps_total_budget=eps_total)
    ck = latest_ckpt(ckdir)
    if ck and os.path.exists(fpf) and open(fpf).read() == fp:
        model = RecurrentPPO.load(ck, env=env); remaining = TOTAL_TIMESTEPS - model.num_timesteps
        print(f'    [{variant}/{seed}] resuming from {model.num_timesteps}/{TOTAL_TIMESTEPS}')
    else:
        shutil.rmtree(ckdir, ignore_errors=True); os.makedirs(ckdir); open(fpf, 'w').write(fp)
        model = RecurrentPPO('MlpLstmPolicy', env, n_steps=128, batch_size=64, learning_rate=3e-4, gamma=0.99, seed=seed, verbose=0,
                             policy_kwargs=dict(lstm_hidden_size=64, n_lstm_layers=1), device='cpu'); remaining = TOTAL_TIMESTEPS
    t0 = time.time()
    if remaining > 0:
        model.learn(total_timesteps=remaining, reset_num_timesteps=False, callback=CheckpointCallback(save_freq=CKPT_FREQ, save_path=ckdir, name_prefix='ckpt'), progress_bar=False)
    model.save(name); open(name + '.fp', 'w').write(fp); shutil.rmtree(ckdir, ignore_errors=True)
    return model, time.time() - t0

## 4. Per-seed sweep

For each seed: refit the tuned target (check against NB3/NB4 clean F1), re-score the **saved NB4 budget-0.1 agent** (reproducibility check against `attack2_seed*_v6.pkl`), then for each new budget train PPO and score PPO, random and PGD on the full attack-class test set.

In [ ]:
class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, n_classes))
    def forward(self, x): return self.net(x)
def pgd_cyber(sur, X, y, eps=0.1, n_iter=10):
    sur.eval(); m = torch.zeros(N_FEAT, device=DEVICE); m[torch.as_tensor(CYBER_IDX, device=DEVICE)] = 1.0; alpha = eps / 4
    yt = torch.from_numpy(y).long().to(DEVICE); X0 = torch.from_numpy(X).float().to(DEVICE); Xa = X0.clone()
    for _ in range(n_iter):
        Xa.requires_grad_(True); g = torch.autograd.grad(F.cross_entropy(sur(Xa), yt), Xa)[0]
        Xa = Xa.detach() + alpha * m * g.sign(); Xa = X0 + torch.clamp(Xa - X0, -eps, eps)
    return Xa.detach().cpu().numpy()

if SMOKE_TEST: TOTAL_TIMESTEPS = 1024
R = {}; INFO = {}; t_all = time.time()          # R[seed][(method, budget)] -> summary
for seed in SEEDS:
    pf = f'attack2b_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = fp_of(best_params, EPS_BOUND, MAX_STEPS, TOTAL_TIMESTEPS, tuple(ALL_BUDGETS), seed, X_atk_test.shape, float(X_atk_test[:5].sum()))
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: R[seed] = saved['R']; INFO[seed] = saved['info']; print(f'seed {seed}: result file found (fingerprint matches) -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True)
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X_train, y_train, verbose=False)
    f1_fit = f1_score(y_test, clf.predict(X_test), average='weighted'); clf.set_params(device='cpu', n_jobs=-1); oracle = Oracle(clf)
    f1_cpu = f1_score(y_test, clf.predict(X_test), average='weighted'); INFO[seed] = {'clean_f1_fit_backend': float(f1_fit), 'clean_f1_cpu_oracle': float(f1_cpu)}
    print(f'  tuned clean F1: fit-backend {f1_fit:.4f} | CPU oracle {f1_cpu:.4f}')
    sur = SurrogateMLP(N_FEAT).to(DEVICE); sur.load_state_dict(torch.load(f'surrogate_mlp_{PIPELINE_VERSION}_seed{seed}.pt', map_location=DEVICE))
    R[seed] = {}
    for b in ALL_BUDGETS:
        Xp = pgd_cyber(sur, X_atk_test, y_atk_test, eps=b); pp = oracle.predict(Xp)
        R[seed][('PGD_cyber', b)] = summarise({'evaded': pp == 0, 'steps': np.where(pp == 0, 1, 0), 'x_final': Xp, 'pred_final': pp, 'clean_pred': None})
        R[seed][('Random', b)] = summarise(run_batched(None, X_atk_test, oracle, b, mode='random', rng=np.random.RandomState(seed + 500)))
    del sur
    # reference budget 0.1: re-score the saved NB4 agent and check it reproduces NB4
    ref_name = f'ppo_attack2_budgeted_{PIPELINE_VERSION}_seed{seed}.zip'
    if os.path.exists(ref_name):
        R[seed][('PPO', REF_BUDGET)] = summarise(run_batched(RecurrentPPO.load(ref_name), X_atk_test, oracle, REF_BUDGET, mode='ppo'))
        nb4f = f'attack2_seed{seed}_{PIPELINE_VERSION}.pkl'
        if os.path.exists(nb4f):
            d = np.abs(joblib.load(nb4f)['R']['PPO_budgeted']['per_class_evasion'] - R[seed][('PPO', REF_BUDGET)]['per_class_evasion']).max()
            INFO[seed]['repro_vs_nb4_max_abs_diff'] = float(d); print(f'  budget 0.1 re-scored from saved NB4 agent; max |diff| vs NB4 per-class evasion = {d:.2e}' + ('' if d < 1e-3 else '   <-- CHECK'))
    else:
        print(f'  {ref_name} not found: budget-0.1 PPO row will be missing for this seed (run NB4 first)')
    for b in BUDGETS:
        agent, tt = get_agent(b, seed, oracle)
        tq = time.time(); R[seed][('PPO', b)] = summarise(run_batched(agent, X_atk_test, oracle, b, mode='ppo'))
        r = R[seed][('PPO', b)]
        print(f'  budget {b:<4} PPO trained {tt / 60:.1f} min | per-class evasion {np.round(r["per_class_evasion"], 3).tolist()} | overall {r["overall_evasion"]:.3f} | random {R[seed][("Random", b)]["overall_evasion"]:.3f} | PGD {R[seed][("PGD_cyber", b)]["overall_evasion"]:.3f}', flush=True)
    joblib.dump({'fp': cfg_fp, 'R': R[seed], 'info': INFO[seed]}, pf)
    print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 5. Evasion versus budget (mean +/- std over seeds, full attack-class test set)

In [ ]:
atk_names = class_names[1:]
def arr(method, b, key): return np.stack([np.asarray(R[s][(method, b)][key]) for s in SEEDS if (method, b) in R[s]])
rows = []
for method in ['PPO', 'Random', 'PGD_cyber']:
    for b in ALL_BUDGETS:
        if method == 'PPO' and b == REF_BUDGET and not all(('PPO', b) in R[s] for s in SEEDS): continue
        a = arr(method, b, 'per_class_evasion'); o = arr(method, b, 'overall_evasion'); sd = lambda x: x.std(0, ddof=1) if len(x) > 1 else np.zeros(x.shape[1:])
        row = {'method': method, 'budget': b}; row.update({c: f'{a[:, i].mean():.3f} +/- {sd(a)[i]:.3f}' for i, c in enumerate(atk_names)})
        row['ALL'] = f'{o.mean():.3f} +/- {(o.std(ddof=1) if len(o) > 1 else 0):.3f}'; row['queries'] = f'{arr(method, b, "mean_queries").mean():.2f}'
        row['added range excess (sigma)'] = f'{arr(method, b, "mean_excess_beyond_train_range").mean():.3f}'; row['rows >0.5 sigma outside'] = f'{arr(method, b, "frac_rows_beyond_range_0.5sigma").mean():.3f}'
        rows.append(row)
sweep_table = pd.DataFrame(rows); pd.set_option('display.width', 250); print(sweep_table.to_string(index=False))
print('\nReference: NB4 unconstrained PPO (ceiling 3.0 sigma): ransomware/reverseshell evasion 1.000, overall 0.996.')

print('\nFirst budget at which PPO mean evasion exceeds 5% / 50% (per class):')
thr = {}
for i, c in enumerate(atk_names):
    m = {b: arr('PPO', b, 'per_class_evasion')[:, i].mean() for b in ALL_BUDGETS if ('PPO', b) in R[SEEDS[0]]}
    t5 = next((b for b in sorted(m) if m[b] > 0.05), None); t50 = next((b for b in sorted(m) if m[b] > 0.50), None); thr[c] = {'gt5pct': t5, 'gt50pct': t50}
    print(f'  {c:<13} >5%: {t5}   >50%: {t50}   (PPO mean by budget: ' + ', '.join(f'{b}:{v:.3f}' for b, v in sorted(m.items())) + ')')

## 6. Does learning matter as the budget grows? (PPO vs random search, paired over seeds)

Per-class p-values are Holm-adjusted over the 5 classes within each budget. With n = 5 seeds and rates that are exactly 0 or 1, a zero-variance comparison is reported as p = 1 only when the means are equal.

In [ ]:
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rank, i in enumerate(o): run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj
def ptest(A, B):
    d = A - B
    if d.std(ddof=1) == 0: return 1.0 if d.mean() == 0 else 0.0
    return float(stats.ttest_rel(A, B).pvalue)
stat_rows = []
if len(SEEDS) > 1:
    for b in ALL_BUDGETS:
        if not all(('PPO', b) in R[s] for s in SEEDS): continue
        for ref in ['Random', 'PGD_cyber']:
            A, B = arr('PPO', b, 'per_class_evasion'), arr(ref, b, 'per_class_evasion'); raw = np.array([ptest(A[:, i], B[:, i]) for i in range(5)]); adj = holm(raw)
            oa, ob = arr('PPO', b, 'overall_evasion'), arr(ref, b, 'overall_evasion')
            for i, c in enumerate(atk_names): stat_rows.append({'budget': b, 'PPO minus': ref, 'class': c, 'diff': f'{(A - B)[:, i].mean():+.3f}', 'p_raw': f'{raw[i]:.4f}', 'p_holm': f'{adj[i]:.4f}'})
            stat_rows.append({'budget': b, 'PPO minus': ref, 'class': 'ALL', 'diff': f'{(oa - ob).mean():+.3f}', 'p_raw': f'{ptest(oa, ob):.4f}', 'p_holm': '-'})
    stat_table = pd.DataFrame(stat_rows); print(stat_table.to_string(index=False))
else:
    stat_table = pd.DataFrame(); print('single seed: paired tests skipped')

# burst-clustered 95% CI of PPO overall evasion at each budget
rng = np.random.RandomState(42); ub = np.unique(test_burst); bi = np.searchsorted(ub, test_burst); nb_ = len(ub); n_atk = np.bincount(bi, minlength=nb_).astype(float); ci = {}
idx = rng.randint(0, nb_, size=(2000, nb_))
for b in ALL_BUDGETS:
    if not all(('PPO', b) in R[s] for s in SEEDS): continue
    boots = [np.bincount(bi, weights=R[s][('PPO', b)]['evaded'].astype(float), minlength=nb_)[idx].sum(1) / n_atk[idx].sum(1) for s in SEEDS]
    ci[b] = (float(arr('PPO', b, 'overall_evasion').mean()), *np.percentile(np.mean(boots, 0), [2.5, 97.5]))
print('\nBurst-clustered 95% CI, PPO overall evasion (pooled over seeds):')
for b, (pt, lo, hi) in ci.items(): print(f'  budget {b:<4} {pt:.4f}  [{lo:.4f}, {hi:.4f}]')

## 7. Figure

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(19, 5)); cols = plt.cm.tab10(np.arange(5))
for i, c in enumerate(atk_names):
    for method, ls in [('PPO', '-'), ('Random', '--'), ('PGD_cyber', ':')]:
        bs = [b for b in ALL_BUDGETS if all((method, b) in R[s] for s in SEEDS)]; A = np.stack([arr(method, b, 'per_class_evasion')[:, i] for b in bs])
        ax[0].plot(bs, A.mean(1), ls, color=cols[i], marker='o', ms=3, label=f'{c} {method}' if method == 'PPO' else None)
        if method == 'PPO' and len(SEEDS) > 1: ax[0].fill_between(bs, A.mean(1) - A.std(1, ddof=1), A.mean(1) + A.std(1, ddof=1), color=cols[i], alpha=.12)
ax[0].set_xscale('log'); ax[0].set_xlabel('cumulative budget (sigma, log)'); ax[0].set_ylabel('evasion rate'); ax[0].set_title('Per-class evasion vs budget (solid PPO, dashed random, dotted PGD)'); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
for method, ls in [('PPO', '-'), ('Random', '--'), ('PGD_cyber', ':')]:
    bs = [b for b in ALL_BUDGETS if all((method, b) in R[s] for s in SEEDS)]; o = np.stack([arr(method, b, 'overall_evasion') for b in bs])
    ax[1].errorbar(bs, o.mean(1), yerr=o.std(1, ddof=1) if len(SEEDS) > 1 else None, fmt=ls, marker='o', capsize=3, label=method)
ax[1].set_xscale('log'); ax[1].set_xlabel('cumulative budget (sigma, log)'); ax[1].set_ylabel('overall evasion'); ax[1].set_title('Overall evasion vs budget'); ax[1].legend(); ax[1].grid(alpha=.3)
bs = [b for b in ALL_BUDGETS if all(('PPO', b) in R[s] for s in SEEDS)]
ax[2].plot(bs, [arr('PPO', b, 'frac_rows_beyond_range_0.5sigma').mean() for b in bs], marker='o', label='rows pushed >0.5 sigma beyond training range')
ax[2].plot(bs, [arr('PPO', b, 'mean_excess_beyond_train_range').mean() for b in bs], marker='s', label='mean added excess (sigma)')
ax[2].set_xscale('log'); ax[2].set_xlabel('cumulative budget (sigma, log)'); ax[2].set_title('Realism: how far outside the training range (PPO)'); ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'attack2b_budget_sweep_{PIPELINE_VERSION}.png', dpi=150); plt.show()

## 8. Save

In [ ]:
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'budgets': ALL_BUDGETS, 'class_names': class_names,
             'per_seed': {s: {f'{k[0]}|{k[1]}': {kk: vv for kk, vv in v.items() if kk != 'evaded'} for k, v in R[s].items()} for s in SEEDS}, 'info': INFO,
             'sweep_table': sweep_table, 'stat_table': stat_table, 'thresholds': thr, 'burst_ci_ppo': {str(b): v for b, v in ci.items()}},
            f'attack2b_results_{PIPELINE_VERSION}.pkl')
sweep_table.to_csv(f'attack2b_sweep_table_{PIPELINE_VERSION}.csv', index=False); stat_table.to_csv(f'attack2b_paired_tests_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB4b', 'pipeline_version': PIPELINE_VERSION, 'backend': BACKEND_NAME, 'xgboost': xgb.__version__, 'torch': torch.__version__, 'sb3_contrib': sb3_contrib.__version__,
           'seeds': SEEDS, 'budgets': ALL_BUDGETS, 'timesteps': TOTAL_TIMESTEPS, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb4b_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack2b_results_{PIPELINE_VERSION}.pkl', f'attack2b_sweep_table_{PIPELINE_VERSION}.csv', f'attack2b_paired_tests_{PIPELINE_VERSION}.csv', f'attack2b_budget_sweep_{PIPELINE_VERSION}.png', f'nb4b_run_manifest_{PIPELINE_VERSION}.json'] \
          + [f'ppo_attack2_budget{str(b).replace(".", "p")}_{PIPELINE_VERSION}_seed{s}.zip' for b in BUDGETS for s in SEEDS]:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)